In [ ]:
import torch

num = "01"
data = torch.load(f"adv_first_batch_epoch0{num}0.pt",weights_only=False)
data["y"].shape

In [8]:
data.keys()

dict_keys(['epoch', 'x_adv', 'y'])

In [3]:
data["y"].shape

torch.Size([2, 256, 256, 10])

In [9]:
data["x_adv"].shape

torch.Size([2, 256, 256, 10])

In [4]:
import os
from pathlib import Path
import torch
import matplotlib.pyplot as plt

def plot_first_frame_heatmaps(
    folder: str = ".",
    nums = range(1, 13),
    file_prefix: str = "adv_first_batch_epoch0",
    save_prefix: str = "heatmap_pair_epoch",
    cmap: str = "viridis",
    dpi: int = 150,
    show: bool = False,
):
    """
    从 adv_first_batch_epoch0{num}0.pt 读取 data['y']，形状 (B,H,W,T)，
    对每个 num 画一张包含两个 heatmap 的图（分别对应 batch=0 和 batch=1），
    取最后一维的第 0 帧：y[b, :, :, 0]。

    Parameters
    ----------
    folder : str
        存放 .pt 文件与输出图片的目录。
    nums : iterable
        序号（1..12）。文件名形如 adv_first_batch_epoch0{num:02d}0.pt。
    file_prefix : str
        输入 .pt 文件的前缀。
    save_prefix : str
        输出图片文件的前缀。
    cmap : str
        matplotlib colormap 名称。
    dpi : int
        保存图片的分辨率。
    show : bool
        是否在生成时也显示图像（notebook/本地调试用）。

    Returns
    -------
    saved_paths : list[str]
        成功保存的图片路径列表。
    """
    folder = Path(folder)
    saved = []

    for n in nums:
        num2 = f"{n:02d}"
        fname = folder / f"{file_prefix}{num2}0.pt"
        if not fname.exists():
            print(f"[skip] file not found: {fname}")
            continue

        data = torch.load(fname, weights_only=False)
        if "y" not in data:
            print(f"[skip] 'y' not in file: {fname}")
            continue

        y = data["y"]  # expect shape (B,H,W,T)
        if not isinstance(y, torch.Tensor) or y.ndim != 4:
            print(f"[skip] unexpected y shape in {fname}: {getattr(y, 'shape', None)}")
            continue

        B, H, W, T = y.shape
        if B < 2 or T < 1:
            print(f"[skip] need at least B=2, T>=1 in {fname}, got {y.shape}")
            continue

        # 取第 0 帧
        frame0_b0 = y[0, :, :, 0].detach().cpu().numpy()
        frame0_b1 = y[1, :, :, 0].detach().cpu().numpy()

        # 共享色标范围，便于对比
        vmin = min(frame0_b0.min(), frame0_b1.min())
        vmax = max(frame0_b0.max(), frame0_b1.max())

        fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        for ax, arr, b in zip(axes, (frame0_b0, frame0_b1), (0, 1)):
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
            ax.set_title(f"epoch 0{num2}0 | batch={b} | y[..., 0]")
            ax.set_xlabel("W")
            ax.set_ylabel("H")
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.set_ylabel("value", rotation=270, labelpad=12)

            # 让坐标轴更清爽
            ax.set_xticks([])
            ax.set_yticks([])

        out_path = folder / f"{save_prefix}0{num2}0_f0.png"
        fig.suptitle(f"{fname.name}  —  first frame heatmaps (B=2)")
        fig.savefig(out_path, dpi=dpi)
        if show:
            plt.show()
        else:
            plt.close(fig)

        print(f"[saved] {out_path}")
        saved.append(str(out_path))

    return saved

# 用法：
saved_imgs = plot_first_frame_heatmaps(folder=".", nums=range(1,13))


[saved] heatmap_pair_epoch0010_f0.png
[saved] heatmap_pair_epoch0020_f0.png
[saved] heatmap_pair_epoch0030_f0.png
[saved] heatmap_pair_epoch0040_f0.png
[saved] heatmap_pair_epoch0050_f0.png
[saved] heatmap_pair_epoch0060_f0.png
[saved] heatmap_pair_epoch0070_f0.png
[saved] heatmap_pair_epoch0080_f0.png
[saved] heatmap_pair_epoch0090_f0.png
[saved] heatmap_pair_epoch0100_f0.png
[saved] heatmap_pair_epoch0110_f0.png
[saved] heatmap_pair_epoch0120_f0.png


In [11]:
import os
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np

def plot_all_10_frames_grid_stacked(
    folder: str = ".",
    nums = range(1, 13),
    file_prefix: str = "adv_first_batch_epoch0",
    save_prefix: str = "heatmap_grid_epoch",
    cmap: str = "viridis",
    dpi: int = 150,
    show: bool = False,
    figsize_per_cell: float = 2.4,  # per-subplot size
):
    """
    For each adv_first_batch_epoch0{num}0.pt, plot a single 4x5 figure:
      - Rows 0-1: batch=0, frames t=0..9 (2 rows x 5 cols)
      - Rows 2-3: batch=1, frames t=0..9 (2 rows x 5 cols)

    Uses a unified vmin/vmax across both batches' 10 frames for fair comparison.
    """
    folder = Path(folder)
    saved = []

    for n in nums:
        num2 = f"{n:02d}"
        fname = folder / f"{file_prefix}{num2}0.pt"
        if not fname.exists():
            print(f"[skip] file not found: {fname}")
            continue

        data = torch.load(fname, weights_only=False)
        if "y" not in data:
            print(f"[skip] 'y' not in file: {fname}")
            continue

        y = data["y"]  # expect (B,H,W,T)
        if not isinstance(y, torch.Tensor) or y.ndim != 4:
            print(f"[skip] unexpected y shape in {fname}: {getattr(y, 'shape', None)}")
            continue

        B, H, W, T = y.shape
        if B < 2 or T < 10:
            print(f"[skip] need at least B=2, T>=10 in {fname}, got {y.shape}")
            continue

        # unified color scale across both batches and 10 frames
        y_np = y[:2, :, :, :10].detach().cpu().numpy()
        vmin = float(np.min(y_np))
        vmax = float(np.max(y_np))

        rows, cols = 4, 5
        fig_w, fig_h = cols * figsize_per_cell, rows * figsize_per_cell
        fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), constrained_layout=True)
        axes = np.asarray(axes).reshape(rows, cols)

        ims = []
        # batch 0 on rows 0-1
        for t in range(10):
            r = (t // cols)        # 0 or 1
            c = (t % cols)
            ax = axes[r, c]
            arr = y[0, :, :, t].detach().cpu().numpy()
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
            ims.append(im)
            ax.set_title(f"b=0, t={t}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])

        # batch 1 on rows 2-3
        for t in range(10):
            r = 2 + (t // cols)    # 2 or 3
            c = (t % cols)
            ax = axes[r, c]
            arr = y[1, :, :, t].detach().cpu().numpy()
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
            ims.append(im)
            ax.set_title(f"b=1, t={t}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])

        # single global colorbar
        cbar = fig.colorbar(ims[0], ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
        cbar.ax.set_ylabel("value", rotation=270, labelpad=12)

        fig.suptitle(f"{fname.name} — 4×5 grid (top: b=0, bottom: b=1)", fontsize=11)
        out_path = folder / f"{save_prefix}0{num2}0_both_t0to9.png"
        fig.savefig(out_path, dpi=dpi)
        if show:
            plt.show()
        else:
            plt.close(fig)

        print(f"[saved] {out_path}")
        saved.append(str(out_path))

    return saved

# 用法：
saved_imgs = plot_all_10_frames_grid_stacked(folder=".", nums=range(1,13))

[saved] heatmap_grid_epoch0010_both_t0to9.png
[saved] heatmap_grid_epoch0020_both_t0to9.png
[saved] heatmap_grid_epoch0030_both_t0to9.png
[saved] heatmap_grid_epoch0040_both_t0to9.png
[saved] heatmap_grid_epoch0050_both_t0to9.png
[saved] heatmap_grid_epoch0060_both_t0to9.png
[saved] heatmap_grid_epoch0070_both_t0to9.png
[saved] heatmap_grid_epoch0080_both_t0to9.png
[saved] heatmap_grid_epoch0090_both_t0to9.png
[saved] heatmap_grid_epoch0100_both_t0to9.png
[saved] heatmap_grid_epoch0110_both_t0to9.png
[saved] heatmap_grid_epoch0120_both_t0to9.png


In [12]:
import os
from pathlib import Path
import torch
import matplotlib.pyplot as plt

def plot_first_frame_heatmaps(
    folder: str = ".",
    nums = range(1, 13),
    file_prefix: str = "adv_first_batch_epoch0",
    save_prefix: str = "heatmap_pair_epoch",
    cmap: str = "viridis",
    dpi: int = 150,
    show: bool = False,
):
    """
    从 adv_first_batch_epoch0{num}0.pt 读取 data['x_adv']，形状 (B,H,W,T_in)，
    对每个 num 画一张包含两个 heatmap 的图（分别对应 batch=0 和 batch=1），
    取最后一维的第 0 帧：x_adv[b, :, :, 0]。

    Parameters
    ----------
    folder : str
        存放 .pt 文件与输出图片的目录。
    nums : iterable
        序号（1..12）。文件名形如 adv_first_batch_epoch0{num:02d}0.pt。
    file_prefix : str
        输入 .pt 文件的前缀。
    save_prefix : str
        输出图片文件的前缀。
    cmap : str
        matplotlib colormap 名称。
    dpi : int
        保存图片的分辨率。
    show : bool
        是否在生成时也显示图像（notebook/本地调试用）。

    Returns
    -------
    saved_paths : list[str]
        成功保存的图片路径列表。
    """
    folder = Path(folder)
    saved = []

    for n in nums:
        num2 = f"{n:02d}"
        fname = folder / f"{file_prefix}{num2}0.pt"
        if not fname.exists():
            print(f"[skip] file not found: {fname}")
            continue

        data = torch.load(fname, weights_only=False)
        if "x_adv" not in data:
            print(f"[skip] 'x_adv' not in file: {fname}")
            continue

        x_adv = data["x_adv"]  # expect shape (B,H,W,T_in)
        if not isinstance(x_adv, torch.Tensor) or x_adv.ndim != 4:
            print(f"[skip] unexpected x_adv shape in {fname}: {getattr(x_adv, 'shape', None)}")
            continue

        B, H, W, T_in = x_adv.shape
        if B < 2 or T_in < 1:
            print(f"[skip] need at least B=2, T_in>=1 in {fname}, got {x_adv.shape}")
            continue

        # 取第 0 帧
        frame0_b0 = x_adv[0, :, :, 0].detach().cpu().numpy()
        frame0_b1 = x_adv[1, :, :, 0].detach().cpu().numpy()

        # 共享色标范围，便于对比
        vmin = min(frame0_b0.min(), frame0_b1.min())
        vmax = max(frame0_b0.max(), frame0_b1.max())

        fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        for ax, arr, b in zip(axes, (frame0_b0, frame0_b1), (0, 1)):
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
            ax.set_title(f"epoch 0{num2}0 | batch={b} | x_adv[..., 0]")
            ax.set_xlabel("W")
            ax.set_ylabel("H")
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.set_ylabel("value", rotation=270, labelpad=12)

            ax.set_xticks([])
            ax.set_yticks([])

        out_path = folder / f"{save_prefix}0{num2}0_xadv_f0.png"
        fig.suptitle(f"{fname.name}  —  first frame heatmaps from x_adv (B=2)")
        fig.savefig(out_path, dpi=dpi)
        if show:
            plt.show()
        else:
            plt.close(fig)

        print(f"[saved] {out_path}")
        saved.append(str(out_path))

    return saved

# 用法：
saved_imgs = plot_first_frame_heatmaps(folder=".", nums=range(1,13))


[saved] heatmap_pair_epoch0010_xadv_f0.png
[saved] heatmap_pair_epoch0020_xadv_f0.png
[saved] heatmap_pair_epoch0030_xadv_f0.png
[saved] heatmap_pair_epoch0040_xadv_f0.png
[saved] heatmap_pair_epoch0050_xadv_f0.png
[saved] heatmap_pair_epoch0060_xadv_f0.png
[saved] heatmap_pair_epoch0070_xadv_f0.png
[saved] heatmap_pair_epoch0080_xadv_f0.png
[saved] heatmap_pair_epoch0090_xadv_f0.png
[saved] heatmap_pair_epoch0100_xadv_f0.png
[saved] heatmap_pair_epoch0110_xadv_f0.png
[saved] heatmap_pair_epoch0120_xadv_f0.png


In [13]:
import os
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np

def plot_all_10_frames_grid_stacked(
    folder: str = ".",
    nums = range(1, 13),
    file_prefix: str = "adv_first_batch_epoch0",
    save_prefix: str = "heatmap_grid_epoch",
    cmap: str = "viridis",
    dpi: int = 150,
    show: bool = False,
    figsize_per_cell: float = 2.4,  # per-subplot size
):
    """
    For each adv_first_batch_epoch0{num}0.pt, plot a single 4x5 figure using data['x_adv']:
      - Rows 0-1: batch=0, frames t=0..9 (2 rows x 5 cols)
      - Rows 2-3: batch=1, frames t=0..9 (2 rows x 5 cols)
    Uses a unified vmin/vmax across both batches' 10 frames for fair comparison.
    """
    folder = Path(folder)
    saved = []

    for n in nums:
        num2 = f"{n:02d}"
        fname = folder / f"{file_prefix}{num2}0.pt"
        if not fname.exists():
            print(f"[skip] file not found: {fname}")
            continue

        data = torch.load(fname, weights_only=False)
        if "x_adv" not in data:
            print(f"[skip] 'x_adv' not in file: {fname}")
            continue

        x_adv = data["x_adv"]  # expect (B,H,W,T_in)
        if not isinstance(x_adv, torch.Tensor) or x_adv.ndim != 4:
            print(f"[skip] unexpected x_adv shape in {fname}: {getattr(x_adv, 'shape', None)}")
            continue

        B, H, W, T_in = x_adv.shape
        if B < 2 or T_in < 10:
            print(f"[skip] need at least B=2, T_in>=10 in {fname}, got {x_adv.shape}")
            continue

        # unified color scale across both batches and first 10 frames
        x_np = x_adv[:2, :, :, :10].detach().cpu().numpy()
        vmin = float(np.min(x_np))
        vmax = float(np.max(x_np))

        rows, cols = 4, 5
        fig_w, fig_h = cols * figsize_per_cell, rows * figsize_per_cell
        fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), constrained_layout=True)
        axes = np.asarray(axes).reshape(rows, cols)

        ims = []
        # batch 0 on rows 0-1
        for t in range(10):
            r = (t // cols)        # 0 or 1
            c = (t % cols)
            ax = axes[r, c]
            arr = x_adv[0, :, :, t].detach().cpu().numpy()
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
            ims.append(im)
            ax.set_title(f"b=0, t={t}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])

        # batch 1 on rows 2-3
        for t in range(10):
            r = 2 + (t // cols)    # 2 or 3
            c = (t % cols)
            ax = axes[r, c]
            arr = x_adv[1, :, :, t].detach().cpu().numpy()
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper")
            ims.append(im)
            ax.set_title(f"b=1, t={t}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])

        # single global colorbar
        cbar = fig.colorbar(ims[0], ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
        cbar.ax.set_ylabel("value", rotation=270, labelpad=12)

        fig.suptitle(f"{fname.name} — x_adv 4×5 grid (top: b=0, bottom: b=1)", fontsize=11)
        out_path = folder / f"{save_prefix}0{num2}0_xadv_both_t0to9.png"
        fig.savefig(out_path, dpi=dpi)
        if show:
            plt.show()
        else:
            plt.close(fig)

        print(f"[saved] {out_path}")
        saved.append(str(out_path))

    return saved

# 用法：
saved_imgs = plot_all_10_frames_grid_stacked(folder=".", nums=range(1,13))


[saved] heatmap_grid_epoch0010_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0020_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0030_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0040_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0050_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0060_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0070_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0080_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0090_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0100_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0110_xadv_both_t0to9.png
[saved] heatmap_grid_epoch0120_xadv_both_t0to9.png
